# Point cloud registration: the PCL pipeline (FPFH, RANSAC, ICP)

*graphics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/point_cloud_registration)

Two partial 3D scans of the same object, taken from two viewpoints and moved by an unknown rigid motion, are aligned the way the Point Cloud Library (PCL) does it, written as batched PyTorch: voxel-grid downsampling, statistical outlier removal, normals from neighbourhood covariances, FPFH descriptors, RANSAC over feature matches (thousands of Kabsch fits at once) and ICP refinement, point-to-point and point-to-plane, from several starts. The result is checked against the known motion: rotation error in degrees, translation error and RMSE.

**Vector form**

$$R^\ast, \mathbf t^\ast = \arg\min_{R \in SO(3),\,\mathbf t}\ \sum_n w_n \big\lVert R\,\mathbf p_n + \mathbf t - \mathbf q_n \big\rVert^2 \quad\text{(Kabsch)},\qquad \min_{\boldsymbol\omega,\,\delta\mathbf t}\ \sum_n w_n\big[(\mathbf p_n - \mathbf q_n)\cdot\mathbf n_n + (\mathbf p_n\times\mathbf n_n)\cdot\boldsymbol\omega + \mathbf n_n\cdot\delta\mathbf t\big]^2 \quad\text{(point-to-plane)}$$

**Index (tensor) form**

$$H_{bij} = w_{bn}\,\bar p_{bni}\,\bar q_{bnj},\qquad H_b = U_b S_b V_b^{\mathsf T},\qquad R_b = V_b\,\mathrm{diag}(1, 1, \det V_bU_b^{\mathsf T})\,U_b^{\mathsf T},\qquad C_{nij} = \tfrac1K\,\bar x_{nki}\,\bar x_{nkj},\qquad A_{bn\cdot} = [\,\mathbf x_{bn}\times\mathbf n_{bn},\ \mathbf n_{bn}\,]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $y_{bni} = R_{bij}x_{bnj} + t_{bi}$ | `einsum('...ij,...nj->...ni', R, X) + t[..., None, :]` | (B, N, 3) | one rigid motion per batch member $b$ (RANSAC hypothesis or ICP start), $j$ summed |
| $d_{nm} = \lVert\mathbf x_n - \mathbf y_m\rVert$ | `torch.cdist(X[chunk], Y).topk(k, largest=False)` | (N, k) | $k$ nearest neighbours, one chunk of rows of the distance matrix at a time |
| $\lfloor \mathbf x_n / s\rfloor$ | `unique(floor(X / s), return_inverse=True); index_add_` | (V, 3) | VoxelGrid: scatter-mean of the points that share a voxel key |
| $m_n \le \mu + \alpha\sigma$ | `statistical_outlier_mask(X, k, alpha)` | (N,) | StatisticalOutlierRemoval: mean neighbour distance against its mean and standard deviation over the cloud |
| $C_{nij}$ | `einsum('nki,nkj->nij', d, d) / k; torch.linalg.eigh(C)` | (N, 3, 3) | neighbourhood covariance; the eigenvector of the smallest eigenvalue is the normal, flipped toward the viewpoint |
| $\lambda_0 / (\lambda_0 + \lambda_1 + \lambda_2)$ | `lam[:, 0] / lam.sum(1)` | (N,) | surface variation (PCL curvature): 0 on a plane |
| $(\alpha, \phi, \theta)$ | `pair_features(p, n_p, q, n_q)` | (N, K) each | Darboux-frame angles of a point pair: $\alpha = \mathbf v\cdot\mathbf n_q$, $\phi = \mathbf u\cdot\hat{\mathbf d}$, $\theta = \operatorname{atan2}(\mathbf w\cdot\mathbf n_q, \mathbf u\cdot\mathbf n_q)$ |
| $\mathrm{FPFH}_n$ | `spfh + einsum('nk,nkh->nh', 1 / dist, spfh[idx]) / k` | (N, 33) | three 11-bin histograms; a point's own histogram plus the distance-weighted mean of its neighbours |
| $H_{bij}$ | `einsum('...n,...ni,...nj->...ij', w, P - p_bar, Q - q_bar)` | (B, 3, 3) | Kabsch cross-covariance, one per hypothesis; one batched SVD gives all rotations |
| $s_b = \#\{m : \lVert R_b\mathbf p_m + \mathbf t_b - \mathbf q_m\rVert < \varepsilon\}$ | `(resid < eps).sum(1)` | (B,) | RANSAC score of every hypothesis at once |
| $A^{\mathsf T}WA\,\boldsymbol\xi = -A^{\mathsf T}W\mathbf r$ | `torch.linalg.solve(H, -g)` | (B, 6) | point-to-plane ICP step $\boldsymbol\xi = (\boldsymbol\omega, \delta\mathbf t)$, batched over the starts |
| $\cos\Delta\theta = \tfrac12(\operatorname{tr}R^{\mathsf T}R_{\rm true} - 1)$ | `rotation_error_deg(R, R_true)` | (B,) | rotation error against the known ground truth |

The demo object is Keenan Crane's "Spot" cow (5856 triangles), which its author released into the public domain (CC0 1.0; https://www.cs.cmu.edu/~kmcrane/Projects/ModelRepository/, bundled as data/spot_vertices.csv and data/spot_faces.csv, see data/SOURCES.md; Crane, Pinkall and Schröder 2013, ACM TOG 32(4)). We did not use the Stanford bunny: the Stanford 3D Scanning Repository allows research use and free redistribution, but says the models are not to be used for commercial purposes or appear in a product for sale without permission, which is unclear for an open project that may later have a paid tier. PCL's own test file bunny.pcd is derived from the same Stanford scan and carries no separate data licence (PCL's BSD licence covers its code). The scans are synthetic: dense random samples of the mesh surface, kept only where a scanner sees them (back-face test plus a z-buffer, i.e. hidden-point removal), with Gaussian noise $\sigma$, a few percent uniform outliers, and a random rigid motion (30–60°, half the object size) applied to scan B. The object size is the bounding-box diagonal, scaled to 1. Pipeline (PCL class names in brackets): VoxelGrid; StatisticalOutlierRemoval; NormalEstimation with $k = 15$ neighbours and the normal flipped toward each scan's own viewpoint; FPFH with $k = 25$; mutual nearest matches in descriptor space; RANSAC with 4000 hypotheses, each 3 matches whose triangle edge lengths agree within 10 % (as in PCL's CorrespondenceRejectorPoly), fitted by Kabsch and scored by the inlier count, then refitted on the inliers; ICP from 6 perturbed copies of the RANSAC pose, against the target downsampled at half the voxel size, ignoring pairs further than one voxel. Simplifications: FPFH uses a fixed $k$ neighbourhood instead of a radius, always takes the query point as the source of the Darboux frame (PCL picks the pair order by the normal angles), and normalises each 11-bin block to 100; the voxel grid averages positions only. What the default run shows: about 12000 visible points per scan become about 1650 after the two filters; 365 mutual FPFH matches, of which RANSAC keeps 73 (about 20 %), give a rough pose 6.4° and 0.056 off; point-to-plane ICP then reaches 0.37° and 0.0021 (0.2 % of the object size, RMSE 0.007) within about 5 iterations from every start, while point-to-point is still creeping along the surface after 25 iterations (about 2.1°, 0.018), because nearest neighbours on a partial, resampled surface pull it sideways. The tests check that Kabsch is exact, a motion composed with its inverse is the identity, plane and sphere normals are right, the voxel grid matches a slow reference, far outliers are removed, FPFH is invariant under rigid motion, z-buffer visibility removes the far side of a sphere, RANSAC survives 50 % wrong matches, ICP converges, point-to-plane is faster than point-to-point on a smooth surface, and the full pipeline gets below 2° and 2 % of the object size. These are established methods: ICP (Besl and McKay 1992; point-to-plane: Chen and Medioni 1992), FPFH (Rusu, Blodow and Beetz 2009), PCL (Rusu and Cousins 2011, BSD licence, https://pointclouds.org). The code here is written from the papers, not translated from PCL. Try a larger view angle (less overlap: RANSAC finds fewer inliers and can fail), more outliers, or a coarser voxel.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/point_cloud_registration/main.py
"""Main loop: two partial scans of Spot → VoxelGrid → outlier removal → normals → FPFH → RANSAC → ICP → errors."""

import time
from pathlib import Path

from problems._common.runtime import cli, fast_mode

from . import plot, solve
from .equation import rotation_error_deg

SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    params = dict(params)
    if fast_mode():
        params.update(n_points=12000, hypotheses=1000, icp_iterations=10, starts=3)
    t0 = time.perf_counter()
    out = solve.register(params, SEED)
    elapsed = time.perf_counter() - t0
    tgt, src = out["tgt"], out["src"]
    print(
        f"scan A: {out['n_a']} visible points + {len(out['A']) - out['n_a']} outliers; "
        f"scan B: {out['n_b']} + {len(out['B']) - out['n_b']} (Spot, object size 1, noise σ = {params['noise']:g})"
    )
    print(
        f"VoxelGrid ({params['voxel']:g}): A {len(out['A'])} → {len(tgt['down'])}, B {len(out['B'])} → "
        f"{len(src['down'])};  StatisticalOutlierRemoval: A → {len(tgt['P'])}, B → {len(src['P'])}"
    )
    print(f"normals: median curvature λ0/Σλ = {tgt['curv'].median():.4f};  FPFH descriptors: {tuple(tgt['F'].shape)}")
    R0, t0_, inl = out["ransac"]
    print(
        f"FPFH mutual matches: {len(out['match'][0])};  RANSAC ({params['hypotheses']} hypotheses, batched Kabsch): "
        f"{int(inl.sum())} inliers, rotation error {out['ransac_rot_err']:.2f}°, translation error "
        f"{out['ransac_trans_err']:.4f}"
    )
    for method, res in out["icp"].items():
        rmse = res["hist"]["rmse"]
        best = int(rmse[-1].argmin())
        err = rotation_error_deg(res["R"][best], out["R_true"])
        print(
            f"ICP {method:>14} ({params['starts']} starts × {params['icp_iterations']} iterations): best RMSE "
            f"{rmse[0, best]:.4f} → {rmse[-1, best]:.4f}, rotation error {err:.2f}°, translation error "
            f"{(res['t'][best] - out['t_true']).norm():.4f}"
        )
    print(
        f"result (point-to-plane, best start): rotation error {out['rot_err']:.3f}° (test < 2°), translation error "
        f"{out['trans_err']:.4f} = {100 * out['trans_err'] / out['size']:.2f}% of the object size (test < 2%), "
        f"RMSE {out['rmse']:.4f};  pipeline {elapsed:.1f} s"
    )
    return plot.animate(out, out_dir / "point_cloud_registration.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/point_cloud_registration/equation.py
r"""Geometry → tensor translation: the PCL registration pipeline as batched tensor operations.

A rigid motion maps a point x to R x + t (R a rotation, t a translation). Indices: b = batch (hypotheses or ICP
starts), n = point, k = neighbour, i, j = 3D axes, h = histogram bin.

    rigid motion     y_{bni} = R_{bij} x_{bnj} + t_{bi}                         j summed
    Kabsch           H_{ij} = w_n (p_{ni} - p̄_i)(q_{nj} - q̄_j), H = U S Vᵀ, R = V diag(1, 1, det V Uᵀ) Uᵀ, t = q̄ - R p̄
    k-NN             d_{nm} = |x_n - y_m|  (torch.cdist), then the k smallest per row (topk)
    voxel grid       key_n = ⌊x_n / s⌋,  c_v = mean of the x_n with key_n = v               (scatter-mean)
    outlier removal  m_n = mean_k d_{nk},  keep m_n ≤ mean(m) + α std(m)
    normals          C_{nij} = (1/K) Σ_k (x_{nki} - x̄_{ni})(x_{nkj} - x̄_{nj}),  C v = λ v,  normal = v of the smallest λ
    curvature        σ_n = λ_0 / (λ_0 + λ_1 + λ_2)
    Darboux frame    u = n_p,  v = u × d̂,  w = u × v,  d̂ = (q - p)/|q - p|
    pair features    α = v·n_q,  φ = u·d̂,  θ = atan2(w·n_q, u·n_q)                → binned into 3 × 11 histograms
    point-to-plane   r_n = (R x_n + t - q_n)·n_n ≈ r_n + (x_n × n_n)·ω + n_n·δt   (small rotation ω)
"""

import torch

BINS = 11  # bins per FPFH angle histogram (PCL's FPFHSignature33 = 3 × 11)


def rodrigues(omega: torch.Tensor) -> torch.Tensor:
    """Axis-angle ω (..., 3) → rotation R (..., 3, 3) = I + sin θ K + (1 - cos θ) K², θ = |ω|, K = [ω/θ]×."""
    theta = omega.norm(dim=-1, keepdim=True).clamp(min=1e-15)  # (..., 1)
    k = omega / theta
    zero = torch.zeros_like(k[..., 0])
    K = torch.stack(
        [
            torch.stack([zero, -k[..., 2], k[..., 1]], -1),
            torch.stack([k[..., 2], zero, -k[..., 0]], -1),
            torch.stack([-k[..., 1], k[..., 0], zero], -1),
        ],
        -2,
    )  # (..., 3, 3): K v = k × v
    s, c = torch.sin(theta)[..., None], torch.cos(theta)[..., None]
    eye = torch.eye(3, dtype=omega.dtype, device=omega.device)
    return eye + s * K + (1 - c) * (K @ K)


def transform(R: torch.Tensor, t: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    """y_{bni} = R_{bij} x_{bnj} + t_{bi}   (j summed).   R (..., 3, 3), t (..., 3), X (..., N, 3) → (..., N, 3)."""
    return torch.einsum("...ij,...nj->...ni", R, X) + t[..., None, :]


def compose(R1, t1, R2, t2):
    """(R1, t1) after (R2, t2): x ↦ R1 (R2 x + t2) + t1 = (R1 R2) x + (R1 t2 + t1)."""
    return R1 @ R2, torch.einsum("...ij,...j->...i", R1, t2) + t1


def invert(R, t):
    """The inverse motion: x = Rᵀ (y - t), i.e. (Rᵀ, -Rᵀ t)."""
    Rt = R.transpose(-1, -2)
    return Rt, -torch.einsum("...ij,...j->...i", Rt, t)


def rotation_error_deg(R1: torch.Tensor, R2: torch.Tensor) -> torch.Tensor:
    """Angle of the rotation R1ᵀ R2 in degrees: cos θ = (tr(R1ᵀ R2) - 1) / 2.   (...,)."""
    cos = ((torch.einsum("...ij,...ij->...", R1, R2) - 1) / 2).clamp(-1.0, 1.0)
    return torch.rad2deg(torch.acos(cos))


def kabsch(P: torch.Tensor, Q: torch.Tensor, w: torch.Tensor | None = None):
    """Best rigid motion (least squares) taking the points P onto Q, batched over the leading dimensions.

    P, Q (..., N, 3), weights w (..., N) (default 1). Returns R (..., 3, 3), t (..., 3) minimising Σ w |R p + t - q|².
    """
    if w is None:
        w = torch.ones(P.shape[:-1], dtype=P.dtype, device=P.device)
    w = w / w.sum(-1, keepdim=True).clamp(min=1e-12)  # (..., N), sums to 1
    p_bar = torch.einsum("...n,...ni->...i", w, P)  # weighted centroids (..., 3)
    q_bar = torch.einsum("...n,...ni->...i", w, Q)
    H = torch.einsum("...n,...ni,...nj->...ij", w, P - p_bar[..., None, :], Q - q_bar[..., None, :])  # n summed
    U, _, Vh = torch.linalg.svd(H)
    V = Vh.transpose(-1, -2)
    d = torch.sign(torch.linalg.det(V @ U.transpose(-1, -2)))  # -1 would be a reflection: flip the last axis
    D = torch.diag_embed(torch.stack([torch.ones_like(d), torch.ones_like(d), d], -1))
    R = V @ D @ U.transpose(-1, -2)
    return R, q_bar - torch.einsum("...ij,...j->...i", R, p_bar)


def knn(X: torch.Tensor, Y: torch.Tensor, k: int, chunk: int = 2048):
    """The k nearest points of Y for every point of X: distances and indices (N, k), nearest first.

    torch.cdist builds a chunk of the distance matrix d_{nm} = |x_n - y_m| at a time, so memory stays O(chunk × M).
    """
    out_d, out_i = [], []
    for s in range(0, len(X), chunk):  # a loop over chunks of rows, not over points
        d = torch.cdist(X[s : s + chunk], Y, compute_mode="use_mm_for_euclid_dist")  # (chunk, M)
        if k == 1:
            d, i = d.min(dim=1, keepdim=True)  # same as topk(1), faster
        else:
            d, i = d.topk(k, dim=1, largest=False)
        out_d.append(d)
        out_i.append(i)
    return torch.cat(out_d), torch.cat(out_i)


def voxel_downsample(X: torch.Tensor, size: float):
    """VoxelGrid filter: key_n = ⌊x_n / s⌋, one output point per occupied voxel = the mean of its points.

    Returns the centroids (V, 3) and the number of points in each voxel (V,).
    """
    keys = torch.floor(X / size).long()  # (N, 3) integer voxel coordinates
    _, inverse, counts = torch.unique(keys, dim=0, return_inverse=True, return_counts=True)
    sums = torch.zeros(len(counts), 3, dtype=X.dtype, device=X.device).index_add_(0, inverse, X)  # scatter-add
    return sums / counts[:, None], counts


def statistical_outlier_mask(X: torch.Tensor, k: int = 20, alpha: float = 1.0) -> torch.Tensor:
    """StatisticalOutlierRemoval: keep the points whose mean distance to their k neighbours is at most
    mean + α·std of that quantity over the cloud.   → bool (N,)."""
    d, _ = knn(X, X, k + 1)  # the first neighbour is the point itself (distance 0)
    m = d[:, 1:].mean(1)  # (N,) mean neighbour distance
    return m <= m.mean() + alpha * m.std()


def estimate_normals(X: torch.Tensor, k: int, viewpoint: torch.Tensor):
    """NormalEstimation: the covariance of each point's k neighbours, its smallest eigenvector, oriented toward
    the viewpoint; and the surface variation λ_0 / (λ_0 + λ_1 + λ_2).   → normals (N, 3), curvature (N,)."""
    _, idx = knn(X, X, k)  # (N, k), includes the point itself
    nb = X[idx]  # (N, k, 3)
    centred = nb - nb.mean(1, keepdim=True)
    C = torch.einsum("nki,nkj->nij", centred, centred) / k  # (N, 3, 3), k summed
    lam, vec = torch.linalg.eigh(C)  # eigenvalues ascending
    normals = vec[:, :, 0]  # (N, 3) direction of least spread
    flip = ((viewpoint - X) * normals).sum(1) < 0  # PCL's flipNormalTowardsViewpoint
    normals = torch.where(flip[:, None], -normals, normals)
    curvature = lam[:, 0].clamp(min=0) / lam.sum(1).clamp(min=1e-30)
    return normals, curvature


def pair_features(p, n_p, q, n_q):
    """Darboux-frame angles (α, φ, θ) for point pairs (p, q) with normals; all inputs (..., 3) → three (...)."""
    d = q - p
    dist = d.norm(dim=-1, keepdim=True).clamp(min=1e-15)
    d_hat = d / dist
    u = n_p
    v = torch.linalg.cross(u, d_hat)
    v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-15)
    w = torch.linalg.cross(u, v)
    alpha = (v * n_q).sum(-1)  # ∈ [-1, 1]
    phi = (u * d_hat).sum(-1)  # ∈ [-1, 1]
    theta = torch.atan2((w * n_q).sum(-1), (u * n_q).sum(-1))  # ∈ [-π, π]
    return alpha, phi, theta, dist[..., 0]


def _histogram(values: torch.Tensor, lo: float, hi: float) -> torch.Tensor:
    """(N, k) values → (N, BINS) counts normalised to sum 100 per row (PCL's convention)."""
    b = ((values - lo) / (hi - lo) * BINS).floor().long().clamp(0, BINS - 1)
    h = torch.nn.functional.one_hot(b, BINS).to(values.dtype).sum(1)  # (N, BINS)
    return 100 * h / h.sum(1, keepdim=True)


def fpfh(X: torch.Tensor, N: torch.Tensor, k: int = 20) -> torch.Tensor:
    """Fast Point Feature Histograms (simplified): (N, 33) = three 11-bin histograms of α, φ, θ per point.

    SPFH_p = histograms over the pairs (p, q_k) of p's k neighbours; FPFH_p = SPFH_p + (1/k) Σ_k SPFH_{q_k} / ω_k with
    ω_k = |q_k - p|, each 11-bin block renormalised to 100.
    """
    _, idx = knn(X, X, k + 1)
    idx = idx[:, 1:]  # (N, k) neighbours, self removed
    alpha, phi, theta, dist = pair_features(X[:, None], N[:, None], X[idx], N[idx])  # each (N, k)
    spfh = torch.cat([_histogram(alpha, -1, 1), _histogram(phi, -1, 1), _histogram(theta, -torch.pi, torch.pi)], 1)
    wgt = 1 / dist.clamp(min=1e-12)  # (N, k)
    f = spfh + torch.einsum("nk,nkh->nh", wgt, spfh[idx]) / k  # (N, 33), k summed
    blocks = f.reshape(-1, 3, BINS)
    return (100 * blocks / blocks.sum(-1, keepdim=True)).reshape(-1, 3 * BINS)


def point_to_plane_system(x: torch.Tensor, q: torch.Tensor, n: torch.Tensor, w: torch.Tensor):
    """Linearised point-to-plane least squares for the small motion ξ = (ω, δt), batched over b:
    A_{bn·} = [x × n, n], r_{bn} = (x - q)·n, then H = Aᵀ W A (6 × 6) and g = Aᵀ W r.   → H (B, 6, 6), g (B, 6)."""
    A = torch.cat([torch.linalg.cross(x, n), n], -1)  # (B, N, 6)
    r = ((x - q) * n).sum(-1)  # (B, N)
    return torch.einsum("bn,bni,bnj->bij", w, A, A), torch.einsum("bn,bni,bn->bi", w, A, r)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/point_cloud_registration/tensors.py
"""Build the concrete tensors: the Spot mesh, dense surface samples, two partial scans seen from two viewpoints
(z-buffer hidden-point removal), sensor noise, outliers and the unknown rigid motion of the second scan.

data/spot_vertices.csv and data/spot_faces.csv hold Keenan Crane's "Spot" cow (2930 vertices, 5856 triangles,
spot_triangulated.obj), released by its author into the public domain (CC0 1.0):
https://www.cs.cmu.edu/~kmcrane/Projects/ModelRepository/  ·  Crane, Pinkall & Schröder 2013, ACM TOG 32(4).
"""

import math
from pathlib import Path

import torch

from . import equation

DATA = Path(__file__).parent / "data"
D = torch.float64
EYE_DISTANCE = 2.5  # scanner distance from the object centre (object size = bounding-box diagonal = 1)

DEFAULTS = {
    "n_points": 30000,
    "view_angle": 70.0,
    "noise": 0.003,
    "outliers": 0.03,
    "voxel": 0.02,
    "hypotheses": 4000,
    "icp_iterations": 25,
    "starts": 6,
}


def load_spot() -> tuple[torch.Tensor, torch.Tensor]:
    """Spot's vertices V (Nv, 3), centred and scaled so the bounding-box diagonal is 1, and faces F (Nf, 3)."""
    V = torch.tensor(_read_csv(DATA / "spot_vertices.csv", float), dtype=D)
    F = torch.tensor(_read_csv(DATA / "spot_faces.csv", int), dtype=torch.long)
    lo, hi = V.min(0).values, V.max(0).values
    return (V - (lo + hi) / 2) / (hi - lo).norm(), F


def _read_csv(path: Path, kind) -> list[list]:
    lines = path.read_text(encoding="utf-8").splitlines()[1:]  # skip the header
    return [[kind(v) for v in line.split(",")] for line in lines if line]


def sample_surface(V: torch.Tensor, F: torch.Tensor, n: int, gen: torch.Generator):
    """n random points uniformly on the mesh surface (triangle chosen ∝ area, uniform barycentric coordinates),
    with the triangle's outward normal.   → points (n, 3), normals (n, 3)."""
    a, b, c = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]  # (Nf, 3) corners
    cross = torch.linalg.cross(b - a, c - a)
    area = cross.norm(dim=1)
    tri = torch.multinomial(area, n, replacement=True, generator=gen)  # (n,)
    r1, r2 = torch.rand(n, 1, generator=gen, dtype=D), torch.rand(n, 1, generator=gen, dtype=D)
    s = r1.sqrt()  # (1 - √r1, √r1 (1 - r2), √r1 r2) is uniform on a triangle
    X = (1 - s) * a[tri] + s * (1 - r2) * b[tri] + s * r2 * c[tri]
    return X, cross[tri] / area[tri, None]


def camera_axes(eye: torch.Tensor):
    """Right, up and forward unit vectors of a camera at eye looking at the origin (world y up)."""
    fwd = -eye / eye.norm()
    right = torch.linalg.cross(fwd, torch.tensor([0.0, 1.0, 0.0], dtype=eye.dtype))
    right = right / right.norm()
    return right, torch.linalg.cross(right, fwd), fwd


def visible(X: torch.Tensor, N: torch.Tensor, eye: torch.Tensor, res: int = 160, tol: float = 0.02) -> torch.Tensor:
    """Which points a scanner at eye sees: back-face test N·(eye - X) > 0, then a z-buffer: project into a res × res
    perspective image, keep per pixel the smallest depth (scatter_reduce 'amin'), and keep points within tol of it.
    → bool (n,)."""
    right, up, fwd = camera_axes(eye)
    rel = X - eye
    z = rel @ fwd  # (n,) depth along the viewing direction
    u, v = (rel @ right) / z, (rel @ up) / z  # perspective image coordinates
    half = torch.maximum(u.abs().max(), v.abs().max()) * 1.0001
    px = ((u / half + 1) / 2 * res).long().clamp(0, res - 1)
    py = ((v / half + 1) / 2 * res).long().clamp(0, res - 1)
    pix = py * res + px  # (n,) flat pixel index
    front = ((eye - X) * N).sum(1) > 0
    depth = torch.where(front, z, torch.inf)
    zbuf = torch.full((res * res,), torch.inf, dtype=X.dtype).scatter_reduce(0, pix, depth, "amin")
    return front & (z <= zbuf[pix] + tol)


def viewpoints(view_angle_deg: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Two scanner positions at azimuths ±view_angle/2 around the vertical axis, 20° above the horizon."""
    el = math.radians(20.0)
    eyes = []
    for az in (-view_angle_deg / 2, view_angle_deg / 2):
        a = math.radians(az + 30.0)  # +30°: look at the cow from its front-left side
        eyes.append(EYE_DISTANCE * torch.tensor([math.sin(a) * math.cos(el), math.sin(el), math.cos(a) * math.cos(el)]))
    return eyes[0].to(D), eyes[1].to(D)


def random_rigid(gen: torch.Generator, max_angle_deg: float, max_trans: float):
    """A random rotation by an angle in [max/2, max] about a random axis, and a random translation of length
    max_trans.   → R (3, 3), t (3,)."""
    axis = torch.randn(3, generator=gen, dtype=D)
    angle = math.radians(max_angle_deg) * (0.5 + 0.5 * torch.rand(1, generator=gen, dtype=D))
    t = torch.randn(3, generator=gen, dtype=D)
    return equation.rodrigues(axis / axis.norm() * angle), max_trans * t / t.norm()


def scan(V, F, eye, n_points, noise, outlier_frac, gen) -> tuple[torch.Tensor, int]:
    """One partial scan: dense surface samples, the visible ones, Gaussian noise σ, plus outlier_frac uniform
    outliers in a box twice the object size.   → points (n, 3), number of surface points (they come first)."""
    X, N = sample_surface(V, F, n_points, gen)
    X = X[visible(X, N, eye)]
    X = X + noise * torch.randn(X.shape, generator=gen, dtype=D)
    n_out = int(outlier_frac * len(X))
    out = (torch.rand(n_out, 3, generator=gen, dtype=D) - 0.5) * 2 * (V.max(0).values - V.min(0).values)
    return torch.cat([X, out]), len(X)


def make_scans(params: dict, seed: int) -> dict:
    """Scan A (the target, world frame) and scan B (the source), B moved by an unknown motion (R_gt, t_gt)."""
    gen = torch.Generator().manual_seed(seed)
    V, F = load_spot()
    eye_a, eye_b = viewpoints(params["view_angle"])
    A, n_a = scan(V, F, eye_a, params["n_points"], params["noise"], params["outliers"], gen)
    B_world, n_b = scan(V, F, eye_b, params["n_points"], params["noise"], params["outliers"], gen)
    R_gt, t_gt = random_rigid(gen, 60.0, 0.5)
    B = equation.transform(R_gt, t_gt, B_world)
    eye_b_own = equation.transform(R_gt, t_gt, eye_b[None])[0]  # the scanner knows where it was in its own frame
    return {"A": A, "B": B, "n_a": n_a, "n_b": n_b, "eye_a": eye_a, "eye_b": eye_b_own, "R_gt": R_gt, "t_gt": t_gt}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/point_cloud_registration/solve.py
"""The registration pipeline: VoxelGrid → StatisticalOutlierRemoval → NormalEstimation → FPFH → feature matching →
batched RANSAC (Kabsch per hypothesis) → batched ICP (point-to-point and point-to-plane) from several starts."""

import torch

from problems._common.progress import steps

from . import equation, tensors


def preprocess(X, eye, voxel: float, k_sor: int = 20, alpha: float = 1.0, features: bool = True) -> dict:
    """One scan → downsampled, cleaned points with normals, curvature and (optionally) FPFH descriptors."""
    down, _ = equation.voxel_downsample(X, voxel)
    keep = equation.statistical_outlier_mask(down, k_sor, alpha)
    P = down[keep]
    N, curv = equation.estimate_normals(P, k=15, viewpoint=eye)
    return {"down": down, "P": P, "N": N, "curv": curv, "F": equation.fpfh(P, N, k=25) if features else None}


def match_features(Fs: torch.Tensor, Ft: torch.Tensor):
    """Mutual nearest neighbours in descriptor space: source i ↔ target j when each is the other's best match."""
    _, st = equation.knn(Fs, Ft, 1)  # (Ns, 1)
    _, ts = equation.knn(Ft, Fs, 1)  # (Nt, 1)
    i = torch.arange(len(Fs), device=Fs.device)
    mutual = ts[st[:, 0], 0] == i
    return i[mutual], st[mutual, 0]


def ransac(P: torch.Tensor, Q: torch.Tensor, n_hypotheses: int, eps: float, gen: torch.Generator, chunk: int = 1000):
    """RANSAC over correspondences p_m ↔ q_m (M, 3) each, many hypotheses at once.

    Each hypothesis draws 3 correspondences, is rejected unless the triangle's edge lengths agree within 10 % (PCL's
    CorrespondenceRejectorPoly: a rigid motion preserves lengths), and is fitted by Kabsch; its score is the number
    of correspondences it maps within eps. The best one is refitted on all its inliers.
    Returns R (3, 3), t (3,), inlier mask (M,).
    """
    M = len(P)
    best_score, best = -1, None
    for s in range(0, n_hypotheses, chunk):
        b = min(chunk, n_hypotheses - s)
        idx = torch.randint(0, M, (b, 3), generator=gen)  # (b, 3) sampled correspondences
        p, q = P[idx], Q[idx]  # (b, 3, 3)
        lp = (p - p.roll(1, dims=1)).norm(dim=-1)  # (b, 3) triangle edge lengths
        lq = (q - q.roll(1, dims=1)).norm(dim=-1)
        ok = ((lp / lq.clamp(min=1e-12)).clamp(max=10) - 1).abs().amax(1) < 0.1
        ok &= lp.amin(1) > 1e-6
        R, t = equation.kabsch(p, q)  # (b, 3, 3), (b, 3): one SVD per hypothesis, batched
        resid = (equation.transform(R, t, P.expand(b, M, 3)) - Q).norm(dim=-1)  # (b, M)
        score = torch.where(ok, (resid < eps).sum(1), -1)
        k = int(score.argmax())
        if score[k] > best_score:
            best_score, best = int(score[k]), (R[k], t[k])
    R, t = best
    for _ in range(3):  # refit on the inliers (a few rounds of least squares)
        inl = (equation.transform(R, t, P) - Q).norm(dim=-1) < eps
        R, t = equation.kabsch(P[inl], Q[inl])
    inl = (equation.transform(R, t, P) - Q).norm(dim=-1) < eps
    return R, t, inl


def icp(src, target, normals, R0, t0, iterations: int, method: str, max_dist: float):
    """Iterative closest point, batched over B starting motions.

    src (B, N, 3) or (N, 3); target (M, 3) with normals (M, 3); R0 (B, 3, 3), t0 (B, 3).
    Each iteration: move src, find each point's nearest target point (pairs further than max_dist are ignored), then
      point_to_point: Kabsch on the pairs (Besl & McKay 1992) gives the whole motion again;
      point_to_plane: one linearised least-squares step on (R x + t - q)·n (Chen & Medioni 1992).
    Returns R, t and the history {"R", "t", "rmse"} (entry 0 = start).
    """
    B = len(R0)
    src = src.expand(B, -1, -1)
    R, t = R0.clone(), t0.clone()
    hist = {"R": [R.clone()], "t": [t.clone()], "rmse": []}
    eye6 = torch.eye(6, dtype=src.dtype, device=src.device)
    for _ in steps(iterations, f"ICP {method}"):
        x = equation.transform(R, t, src)  # (B, N, 3)
        d, j = equation.knn(x.reshape(-1, 3), target, 1)
        d, j = d.reshape(B, -1), j.reshape(B, -1)  # (B, N)
        w = (d < max_dist).to(src.dtype)
        hist["rmse"].append(((w * d**2).sum(1) / w.sum(1).clamp(min=1)).sqrt())
        q = target[j]  # (B, N, 3) matched target points
        if method == "point_to_point":
            R, t = equation.kabsch(src, q, w)
        else:
            H, g = equation.point_to_plane_system(x, q, normals[j], w)
            xi = torch.linalg.solve(H + 1e-9 * eye6, -g)  # (B, 6) = (ω, δt)
            dR = equation.rodrigues(xi[:, :3])
            R, t = equation.compose(dR, xi[:, 3:], R, t)  # apply the small motion after the current one
        hist["R"].append(R.clone())
        hist["t"].append(t.clone())
    x = equation.transform(R, t, src)
    d = equation.knn(x.reshape(-1, 3), target, 1)[0].reshape(B, -1)
    w = (d < max_dist).to(src.dtype)
    hist["rmse"].append(((w * d**2).sum(1) / w.sum(1).clamp(min=1)).sqrt())
    hist["rmse"] = torch.stack(hist["rmse"])  # (iterations + 1, B)
    return R, t, hist


def perturbations(R, t, n: int, angle_deg: float, trans: float, gen: torch.Generator):
    """n starting motions: the given one first, then random small rotations and shifts around it."""
    omega = torch.randn(n, 3, generator=gen, dtype=R.dtype)
    omega = omega / omega.norm(dim=1, keepdim=True) * torch.deg2rad(torch.tensor(angle_deg, dtype=R.dtype))
    dt = trans * torch.randn(n, 3, generator=gen, dtype=R.dtype)
    omega[0], dt[0] = 0, 0
    return equation.compose(equation.rodrigues(omega), dt, R.expand(n, 3, 3), t.expand(n, 3))


def register(params: dict, seed: int = 0) -> dict:
    """The whole pipeline on the two Spot scans; errors against the known ground truth."""
    gen = torch.Generator().manual_seed(seed + 100)
    scans = tensors.make_scans(params, seed)
    voxel = params["voxel"]
    tgt = preprocess(scans["A"], scans["eye_a"], voxel)
    src = preprocess(scans["B"], scans["eye_b"], voxel)
    i_s, i_t = match_features(src["F"], tgt["F"])
    R0, t0, inl = ransac(src["P"][i_s], tgt["P"][i_t], params["hypotheses"], 2.5 * voxel, gen)
    Rs, ts = perturbations(R0, t0, params["starts"], 3.0, 0.3 * voxel, gen)
    fine = preprocess(scans["A"], scans["eye_a"], voxel / 2, features=False)  # a denser target for ICP
    icp_out = {}
    for method in ["point_to_point", "point_to_plane"]:
        R, t, hist = icp(src["P"], fine["P"], fine["N"], Rs, ts, params["icp_iterations"], method, voxel)
        icp_out[method] = {"R": R, "t": t, "hist": hist}
    best = icp_out["point_to_plane"]
    b = int(best["hist"]["rmse"][-1].argmin())  # the start that ended with the smallest RMSE
    R_true, t_true = equation.invert(scans["R_gt"], scans["t_gt"])  # the motion that undoes the unknown one
    rot_err = float(equation.rotation_error_deg(best["R"][b], R_true))
    trans_err = float((best["t"][b] - t_true).norm())
    return {
        **scans, "src": src, "tgt": tgt, "fine": fine, "match": (i_s, i_t), "ransac": (R0, t0, inl), "starts": (Rs, ts),
        "icp": icp_out, "best": b, "R_true": R_true, "t_true": t_true, "rot_err": rot_err, "trans_err": trans_err,
        "ransac_rot_err": float(equation.rotation_error_deg(R0, R_true)),
        "ransac_trans_err": float((t0 - t_true).norm()), "rmse": float(best["hist"]["rmse"][-1, b]), "size": 1.0,
    }  # fmt: skip


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/point_cloud_registration/plot.py
"""Animate the registration: two scans start misaligned, the best RANSAC hypothesis snaps them together, ICP slides
them into place (error curves on the right), and a panel shows the normals and the FPFH matches."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d.art3d import Line3DCollection

from problems._common.plotting import save_gif

from . import equation

BLUE, RED, GREEN, GREY, DARK, ORANGE = "#2563eb", "#dc2626", "#16a34a", "#9ca3af", "#374151", "#f59e0b"
HOLD, SNAP = 6, 10  # frames before RANSAC, frames for the RANSAC snap


def _xyz(p: torch.Tensor):
    """World (x, y up, z) → matplotlib (x, z, y) so the cow stands upright."""
    p = p.numpy()
    return p[:, 0], p[:, 2], p[:, 1]


def _log_rotation(R: torch.Tensor) -> torch.Tensor:
    """Axis-angle ω of a rotation (angle < π): |ω| = angle, direction = axis."""
    angle = torch.acos(((torch.trace(R) - 1) / 2).clamp(-1, 1))
    v = torch.stack([R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1]])
    return v * (angle / (2 * torch.sin(angle).clamp(min=1e-12)))


def animate(out: dict, path: Path) -> Path:
    src, tgt = out["src"], out["tgt"]
    R0, t0, inl = out["ransac"]
    b = out["best"]
    plane = out["icp"]["point_to_plane"]["hist"]
    point = out["icp"]["point_to_point"]["hist"]
    n_icp = len(plane["R"]) - 1

    # the motion applied to the source in every frame: identity → RANSAC (interpolated) → ICP iterations
    omega0 = _log_rotation(R0)
    poses = [
        (torch.eye(3, dtype=R0.dtype), torch.zeros(3, dtype=R0.dtype), "1. two scans, unknown relative pose")
    ] * HOLD
    for s in np.linspace(0, 1, SNAP):
        poses.append((equation.rodrigues(s * omega0), s * t0, "2. RANSAC on FPFH matches: best hypothesis"))
    poses += [(R0, t0, "2. RANSAC on FPFH matches: best hypothesis")] * 3
    icp_iter = [-1] * len(poses)
    for k in range(n_icp + 1):
        reps = 3 if k < 4 else 1
        poses += [(plane["R"][k][b], plane["t"][k][b], f"3. point-to-plane ICP, iteration {k}")] * reps
        icp_iter += [k] * reps
    poses += [poses[-1]] * 4
    icp_iter += [n_icp] * 4

    # rotation errors (deg) of every ICP start against the truth, both methods
    def rot_errs(h):
        return torch.stack([equation.rotation_error_deg(Rk, out["R_true"]) for Rk in h["R"]]).numpy()  # (iters+1, B)

    e_plane, e_point = rot_errs(plane), rot_errs(point)
    its = np.arange(n_icp + 1)

    fig = plt.figure(figsize=(10.0, 5.6))
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1.9], width_ratios=[1, 1, 1.05])
    ax3 = fig.add_subplot(gs[:, :2], projection="3d")
    ax_e, ax_r = fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 2])
    ax_m = fig.add_subplot(gs[2, 2], projection="3d")

    # --- main 3D view ---
    A = tgt["P"]
    ax3.scatter(*_xyz(A), s=2, color=BLUE, alpha=0.6, depthshade=False, label="scan A (target)")
    moved = ax3.scatter(*_xyz(src["P"]), s=2, color=RED, alpha=0.6, depthshade=False, label="scan B (source)")
    c, half = A.mean(0), 0.42 * (A.max(0).values - A.min(0).values).max().item()
    lims = [(float(c[i]) - half, float(c[i]) + half) for i in (0, 2, 1)]
    ax3.set_xlim(*lims[0])
    ax3.set_ylim(*lims[1])
    ax3.set_zlim(*lims[2])
    ax3.set_box_aspect((1, 1, 1), zoom=1.0)
    ax3.set_axis_off()
    ax3.legend(fontsize=7, loc="upper left", markerscale=4)
    title = ax3.set_title("", fontsize=10)

    # --- error curves (all starts in grey, the best start in colour) ---
    for ax, data, ylabel in [(ax_e, (e_point, e_plane), "rotation error (°)"),
                             (ax_r, (point["rmse"].numpy(), plane["rmse"].numpy()), "RMSE")]:  # fmt: skip
        for arr, col, name in [(data[0], ORANGE, "point-to-point"), (data[1], GREEN, "point-to-plane")]:
            ax.semilogy(its, np.maximum(arr, 1e-4), color=col, lw=0.6, alpha=0.35)
            ax.semilogy(its, np.maximum(arr[:, b], 1e-4), color=col, lw=1.5, label=name)
        ax.set_xlim(0, n_icp)
        ax.set_ylabel(ylabel, fontsize=7.5)
        ax.tick_params(labelsize=6.5)
        ax.minorticks_off()
    ax_e.axhline(2.0, color=DARK, ls="--", lw=0.8)
    ax_e.text(0.3, 2.0, "2° test", fontsize=6.5, ha="left", va="bottom", color=DARK)
    ax_e.legend(fontsize=6.5, loc="upper right")
    ax_e.set_title(f"ICP from {e_plane.shape[1]} starts vs ground truth", fontsize=8.5)
    ax_r.set_xlabel("ICP iteration", fontsize=7.5)
    cursors = [ax.axvline(0, color=DARK, lw=0.8, alpha=0.0) for ax in (ax_e, ax_r)]

    # --- normals and FPFH matches: source drawn after RANSAC, shifted sideways so the lines are visible ---
    shift = torch.tensor([0.0, 0.0, 0.0], dtype=A.dtype)
    shift[0] = 1.1 * (A[:, 0].max() - A[:, 0].min()).item()
    S = equation.transform(R0, t0, src["P"]) + shift
    NS = src["N"] @ R0.T
    ax_m.scatter(*_xyz(A), s=0.6, color=BLUE, alpha=0.5, depthshade=False)
    ax_m.scatter(*_xyz(S), s=0.6, color=RED, alpha=0.5, depthshade=False)
    gen = torch.Generator().manual_seed(0)
    for P, N, col in [(A, tgt["N"], BLUE), (S, NS, RED)]:
        sel = torch.randperm(len(P), generator=gen)[:250]
        seg = torch.stack([P[sel], P[sel] + 0.05 * N[sel]], 1)[..., [0, 2, 1]]  # (250, 2, 3) short normal lines
        ax_m.add_collection3d(Line3DCollection(seg.numpy(), colors=col, linewidths=0.4))
    i_s, i_t = out["match"]
    order = torch.randperm(len(i_s), generator=gen)[:60]
    lines = torch.stack([S[i_s[order]], A[i_t[order]]], 1)[..., [0, 2, 1]]  # (60, 2, 3) match lines
    good = inl[order].numpy()
    ax_m.add_collection3d(Line3DCollection(lines.numpy()[~good], colors=GREY, linewidths=0.4))
    ax_m.add_collection3d(Line3DCollection(lines.numpy()[good], colors=GREEN, linewidths=0.8))
    ax_m.set_axis_off()
    ax_m.set_title(f"normals and FPFH matches: {int(inl.sum())} of {len(i_s)} RANSAC inliers (green)", fontsize=7.5)
    allm = torch.cat([A, S])
    cm, hm = allm.mean(0), 0.5 * (allm.max(0).values - allm.min(0).values).max().item()
    ax_m.set_xlim(cm[0] - hm, cm[0] + hm)
    ax_m.set_ylim(cm[2] - hm, cm[2] + hm)
    ax_m.set_zlim(cm[1] - hm, cm[1] + hm)
    ax_m.set_box_aspect((1, 1, 1), zoom=1.8)
    ax_m.view_init(elev=25, azim=-90)

    footer = fig.text(0.5, 0.01, "", ha="center", va="bottom", fontsize=8.5)

    def update(f: int):
        R, t, label = poses[f]
        moved._offsets3d = _xyz(equation.transform(R, t, src["P"]))
        ax3.view_init(elev=15, azim=-70 + 1.2 * f)
        title.set_text(label)
        k = icp_iter[f]
        for cur in cursors:
            cur.set_xdata([max(k, 0)] * 2)
            cur.set_alpha(0.0 if k < 0 else 0.8)
        err = float(equation.rotation_error_deg(R, out["R_true"]))
        terr = float((t - out["t_true"]).norm())
        footer.set_text(f"error vs ground truth: rotation {err:.2f}°, translation {terr:.4f} (object size 1)")

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, list(range(len(poses))), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_points': 30000,
    'view_angle': 70.0,
    'noise': 0.003,
    'outliers': 0.03,
    'voxel': 0.02,
    'hypotheses': 4000,
    'icp_iterations': 25,
    'starts': 6
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.point_cloud_registration")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.point_cloud_registration.main")
problem_dir = Path("problems/graphics/point_cloud_registration")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
